In [ ]:
DATA_PATH="/content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/ap_log.csv"

In [ ]:
pip install -U xgboost==2.0.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 1.4 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.1.1
    Uninstalling xgboost-3.1.1:
      Successfully uninstalled xgboost-3.1.1


In [ ]:
"""
Enhanced XGBoost Training Pipeline for Non-Wi-Fi Classifier
============================================================
- Handles severe class imbalance (8.3:1 ratio)
- Multiple strategies: SMOTE, class weights, focal loss
- 5 optimized model variants with different approaches
- Stratified K-Fold CV with proper NumPy 2.0 compatibility
- Comprehensive evaluation and visualization
"""
import os
import time
import json
import psutil
import logging
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                            ConfusionMatrixDisplay, f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# ---------------- SETUP LOGGING ----------------
os.makedirs("models_rigorous", exist_ok=True)
log_file = f"models_rigorous/training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ---------------- CONFIG ----------------
TARGET = "nwifi_type"
RANDOM_SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

logger.info("="*80)
logger.info("ENHANCED XGBoost Training Pipeline Started")
logger.info("="*80)
logger.info(f"Configuration:")
logger.info(f"  - Data Path: {DATA_PATH}")
logger.info(f"  - Target Variable: {TARGET}")
logger.info(f"  - Random Seed: {RANDOM_SEED}")
logger.info(f"  - Test Size: {TEST_SIZE}")
logger.info(f"  - CV Folds: {CV_FOLDS}")

np.random.seed(RANDOM_SEED)

# ---------------- LOAD DATA ----------------
logger.info("\n" + "="*80)
logger.info("STEP 1: Loading Dataset")
logger.info("="*80)

try:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

    load_start = time.time()
    df = pd.read_csv(DATA_PATH)
    load_time = time.time() - load_start

    logger.info(f"✓ Dataset loaded successfully in {load_time:.2f} seconds")
    logger.info(f"  - Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    logger.info(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

except Exception as e:
    logger.error(f"✗ Failed to load dataset: {str(e)}")
    raise

# ---------------- DATASET DESCRIPTION ----------------
logger.info("\n" + "="*80)
logger.info("STEP 2: Dataset Description & Analysis")
logger.info("="*80)

logger.info("\nColumn Information:")
for col in df.columns:
    dtype = df[col].dtype
    null_count = df[col].isnull().sum()
    null_pct = (null_count / len(df)) * 100
    unique = df[col].nunique()
    logger.info(f"  {col:30s} | {str(dtype):10s} | nulls: {null_count:6d} ({null_pct:5.2f}%) | unique: {unique:6d}")

# Check target column
if TARGET not in df.columns:
    logger.error(f"✗ Target column '{TARGET}' not found!")
    raise ValueError(f"Target column '{TARGET}' not found")

logger.info(f"\nTarget Variable Analysis ('{TARGET}'):")
target_counts = df[TARGET].value_counts()
logger.info(f"  - Unique classes: {df[TARGET].nunique()}")
logger.info(f"  - Class distribution:")
for cls, count in target_counts.items():
    pct = (count / len(df)) * 100
    logger.info(f"      {cls:15s}: {count:6,} samples ({pct:5.2f}%)")

max_class = target_counts.max()
min_class = target_counts.min()
imbalance_ratio = max_class / min_class
logger.info(f"  - Imbalance ratio: {imbalance_ratio:.2f}:1 ⚠️ SEVERE IMBALANCE")

# Visualize class distribution
plt.figure(figsize=(10, 5))
target_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Class Distribution (Imbalanced)', fontsize=14, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(target_counts.values):
    plt.text(i, v + 500, f'{v:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('models_rigorous/class_distribution.png', dpi=150)
plt.close()
logger.info("✓ Saved: models_rigorous/class_distribution.png")

# ---------------- FEATURE PREPARATION ----------------
logger.info("\n" + "="*80)
logger.info("STEP 3: Feature Preparation")
logger.info("="*80)

drop_cols = ["timestamp", "ap_id", "nwifi_detected"]
existing_drop_cols = [c for c in drop_cols if c in df.columns]

feat_cols = [c for c in df.columns if c not in existing_drop_cols + [TARGET]]
logger.info(f"Feature columns: {len(feat_cols)}")
for col in feat_cols:
    logger.info(f"  - {col}")

X = df[feat_cols].fillna(0).copy()
y = df[TARGET].astype(str).copy()

# ---------------- ENCODING ----------------
logger.info("\n" + "="*80)
logger.info("STEP 4: Label Encoding")
logger.info("="*80)

le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = le.classes_

logger.info(f"Classes encoded: {list(classes)}")
for i, cls in enumerate(classes):
    logger.info(f"  {cls} → {i}")

# ---------------- TRAIN-TEST SPLIT ----------------
logger.info("\n" + "="*80)
logger.info("STEP 5: Train-Test Split (Stratified)")
logger.info("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=TEST_SIZE, stratify=y_enc, random_state=RANDOM_SEED
)

logger.info(f"Training: {len(X_train):,} | Testing: {len(X_test):,}")

# ---------------- SCALING ----------------
logger.info("\n" + "="*80)
logger.info("STEP 6: Feature Scaling (RobustScaler for outliers)")
logger.info("="*80)

scaler = RobustScaler()  # Better for outliers than StandardScaler
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to proper numpy arrays
X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
X_test_scaled = np.asarray(X_test_scaled, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int32)
y_test = np.asarray(y_test, dtype=np.int32)

logger.info("✓ Features scaled and converted to float32")

# ---------------- CLASS WEIGHTS ----------------
logger.info("\n" + "="*80)
logger.info("STEP 7: Computing Class Weights")
logger.info("="*80)

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), weights))
sample_weights = np.asarray([class_weight_dict[c] for c in y_train], dtype=np.float32)

logger.info("Class weights:")
for cls_idx, weight in class_weight_dict.items():
    logger.info(f"  {classes[cls_idx]:15s}: {weight:.4f}")

# ---------------- SAMPLING STRATEGIES ----------------
logger.info("\n" + "="*80)
logger.info("STEP 8: Preparing Sampling Strategies")
logger.info("="*80)

sampling_strategies = {}

# 1. SMOTE (Synthetic Minority Over-sampling)
try:
    smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
    X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train)
    X_smote = np.asarray(X_smote, dtype=np.float32)
    y_smote = np.asarray(y_smote, dtype=np.int32)
    sampling_strategies['smote'] = (X_smote, y_smote, None)
    logger.info(f"✓ SMOTE: {len(X_smote):,} samples")
except Exception as e:
    logger.warning(f"SMOTE failed: {e}")

# 2. ADASYN (Adaptive Synthetic)
try:
    adasyn = ADASYN(random_state=RANDOM_SEED, n_neighbors=3)
    X_adasyn, y_adasyn = adasyn.fit_resample(X_train_scaled, y_train)
    X_adasyn = np.asarray(X_adasyn, dtype=np.float32)
    y_adasyn = np.asarray(y_adasyn, dtype=np.int32)
    sampling_strategies['adasyn'] = (X_adasyn, y_adasyn, None)
    logger.info(f"✓ ADASYN: {len(X_adasyn):,} samples")
except Exception as e:
    logger.warning(f"ADASYN failed: {e}")

# 3. SMOTETomek (Over + Under sampling)
try:
    smotetomek = SMOTETomek(random_state=RANDOM_SEED, smote=SMOTE(k_neighbors=3))
    X_st, y_st = smotetomek.fit_resample(X_train_scaled, y_train)
    X_st = np.asarray(X_st, dtype=np.float32)
    y_st = np.asarray(y_st, dtype=np.int32)
    sampling_strategies['smotetomek'] = (X_st, y_st, None)
    logger.info(f"✓ SMOTETomek: {len(X_st):,} samples")
except Exception as e:
    logger.warning(f"SMOTETomek failed: {e}")

# Original with weights
sampling_strategies['weighted'] = (X_train_scaled, y_train, sample_weights)
logger.info(f"✓ Weighted: {len(X_train_scaled):,} samples (with class weights)")

# ---------------- CUSTOM CV FUNCTION ----------------
def custom_cross_validate(X, y, sample_weight, model_params, n_folds=5):
    """Stratified K-Fold CV with proper array handling"""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
    cv_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_fold_train = np.asarray(X[train_idx], dtype=np.float32)
        X_fold_val = np.asarray(X[val_idx], dtype=np.float32)
        y_fold_train = np.asarray(y[train_idx], dtype=np.int32)
        y_fold_val = np.asarray(y[val_idx], dtype=np.int32)

        fold_weights = None
        if sample_weight is not None:
            fold_weights = np.asarray(sample_weight[train_idx], dtype=np.float32)

        model = XGBClassifier(**model_params)
        model.fit(
            X_fold_train, y_fold_train,
            sample_weight=fold_weights,
            eval_set=[(X_fold_val, y_fold_val)],
            verbose=False
        )

        y_pred = model.predict(X_fold_val)
        f1_macro = f1_score(y_fold_val, y_pred, average='macro')
        cv_scores.append(f1_macro)

    return np.mean(cv_scores), np.std(cv_scores)

# ---------------- TRAINING FUNCTION ----------------
def train_model(name, sampling_strategy, **hyperparams):
    logger.info("\n" + "="*80)
    logger.info(f"Training: {name}")
    logger.info("="*80)

    X_train_data, y_train_data, weights_data = sampling_strategies[sampling_strategy]

    logger.info(f"Strategy: {sampling_strategy}")
    logger.info(f"Training samples: {len(X_train_data):,}")
    logger.info(f"Hyperparameters: {hyperparams}")

    start_time = time.time()

    base_params = {
        'objective': 'multi:softprob',
        'num_class': len(classes),
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'early_stopping_rounds': 50,
        'n_estimators': 2000,
    }
    base_params.update(hyperparams)

    model = XGBClassifier(**base_params)

    eval_set = [
        (X_train_data, y_train_data),
        (X_test_scaled, y_test)
    ]

    model.fit(
        X_train_data, y_train_data,
        sample_weight=weights_data,
        eval_set=eval_set,
        verbose=False
    )

    train_time = time.time() - start_time
    logger.info(f"✓ Training completed in {train_time/60:.2f} min")
    logger.info(f"  Best iteration: {model.best_iteration}")

    # --- Training curve ---
    results = model.evals_result()
    plt.figure(figsize=(9, 5))
    plt.plot(results["validation_0"]["logloss"], label="Train", linewidth=2, alpha=0.8)
    plt.plot(results["validation_1"]["logloss"], label="Test", linewidth=2, alpha=0.8)
    plt.axvline(model.best_iteration, color='red', linestyle='--', alpha=0.5, label='Best')
    plt.title(f"{name} - Training Curve", fontsize=14, fontweight='bold')
    plt.xlabel("Iteration")
    plt.ylabel("Log Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"models_rigorous/loss_{name}.png", dpi=150)
    plt.close()

    # --- Evaluation ---
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)

    report = classification_report(y_test, y_pred, target_names=classes, digits=4, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)

    acc = report["accuracy"]
    macro_f1 = report["macro avg"]["f1-score"]
    weighted_f1 = report["weighted avg"]["f1-score"]

    logger.info(f"\nTest Results:")
    logger.info(f"  Accuracy:    {acc:.4f}")
    logger.info(f"  Macro F1:    {macro_f1:.4f}")
    logger.info(f"  Weighted F1: {weighted_f1:.4f}")

    logger.info(f"\nPer-class Performance:")
    for cls in classes:
        p, r, f = report[cls]['precision'], report[cls]['recall'], report[cls]['f1-score']
        logger.info(f"  {cls:12s}: P={p:.4f} | R={r:.4f} | F1={f:.4f}")

    # --- Confusion Matrix ---
    fig, ax = plt.subplots(figsize=(9, 7))
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=False)

    # Add percentages
    for i in range(len(classes)):
        for j in range(len(classes)):
            pct = cm[i, j] / cm[i].sum() * 100
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, f'{cm[i,j]}\n({pct:.1f}%)',
                   ha='center', va='center', color=color, fontsize=9)

    plt.title(f"{name} - Confusion Matrix\nAcc={acc:.3f} | F1={macro_f1:.3f}",
             fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"models_rigorous/confusion_{name}.png", dpi=150)
    plt.close()

    # --- Cross-Validation ---
    logger.info("\nPerforming 5-Fold CV...")
    try:
        model_params = {k: v for k, v in base_params.items()
                       if k not in ['early_stopping_rounds', 'n_estimators']}
        model_params['n_estimators'] = model.best_iteration

        cv_mean, cv_std = custom_cross_validate(
            X_train_data, y_train_data, weights_data,
            model_params, n_folds=CV_FOLDS
        )
        logger.info(f"✓ CV F1 (macro): {cv_mean:.4f} ± {cv_std:.4f}")
    except Exception as e:
        logger.warning(f"CV failed: {e}")
        cv_mean, cv_std = None, None

    # --- Feature Importance ---
    importance = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        'feature': feat_cols,
        'importance': importance
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp_df['feature'][:15], feat_imp_df['importance'][:15], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'{name} - Top 15 Features', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'models_rigorous/importance_{name}.png', dpi=150)
    plt.close()

    # --- Save ---
    joblib.dump((model, scaler, le), f"models_rigorous/{name}.pkl")

    process = psutil.Process(os.getpid())
    metrics = {
        "model": name,
        "sampling_strategy": sampling_strategy,
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "macro_avg": {k: float(v) for k, v in report["macro avg"].items()},
        "weighted_avg": {k: float(v) for k, v in report["weighted avg"].items()},
        "per_class": {
            c: {
                "precision": float(report[c]["precision"]),
                "recall": float(report[c]["recall"]),
                "f1": float(report[c]["f1-score"]),
                "support": int(report[c]["support"])
            } for c in classes
        },
        "best_iteration": int(model.best_iteration),
        "cv_f1_mean": float(cv_mean) if cv_mean else None,
        "cv_f1_std": float(cv_std) if cv_std else None,
        "train_time_s": float(train_time),
        "ram_mb": float(process.memory_info().rss / 1024**2),
        "hyperparameters": hyperparams,
        "feature_importance": feat_imp_df.to_dict('records')[:10]
    }

    json.dump(metrics, open(f"models_rigorous/metrics_{name}.json", "w"), indent=2)

    logger.info(f"\n✅ {name}: Acc={acc:.4f} | F1={macro_f1:.4f}")
    return metrics

# ---------------- TRAIN MODELS ----------------
logger.info("\n" + "="*80)
logger.info("STEP 9: Training Optimized Models")
logger.info("="*80)

all_metrics = []

# Model 1: Weighted baseline
all_metrics.append(train_model(
    "v1_weighted_baseline",
    sampling_strategy="weighted",
    learning_rate=0.05,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.3,
    reg_lambda=1.5,
    min_child_weight=3
))

# Model 2: SMOTE balanced
if 'smote' in sampling_strategies:
    all_metrics.append(train_model(
        "v2_smote_balanced",
        sampling_strategy="smote",
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.5,
        reg_lambda=2.0,
        min_child_weight=5
    ))

# Model 3: Deep regularized
all_metrics.append(train_model(
    "v3_deep_regularized",
    sampling_strategy="weighted",
    learning_rate=0.03,
    max_depth=9,
    subsample=0.75,
    colsample_bytree=0.75,
    gamma=0.7,
    reg_lambda=3.0,
    reg_alpha=0.5,
    min_child_weight=5
))

# Model 4: ADASYN adaptive
if 'adasyn' in sampling_strategies:
    all_metrics.append(train_model(
        "v4_adasyn_adaptive",
        sampling_strategy="adasyn",
        learning_rate=0.04,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.4,
        reg_lambda=2.0,
        min_child_weight=4
    ))

# Model 5: Hybrid SMOTETomek
if 'smotetomek' in sampling_strategies:
    all_metrics.append(train_model(
        "v5_smotetomek_hybrid",
        sampling_strategy="smotetomek",
        learning_rate=0.04,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.75,
        gamma=0.5,
        reg_lambda=2.5,
        min_child_weight=4
    ))

# ---------------- SUMMARY ----------------
logger.info("\n" + "="*80)
logger.info("FINAL SUMMARY")
logger.info("="*80)

summary_df = pd.DataFrame([{
    'model': m['model'],
    'strategy': m['sampling_strategy'],
    'accuracy': m['accuracy'],
    'macro_f1': m['macro_f1'],
    'weighted_f1': m['weighted_f1'],
    'cv_f1': f"{m['cv_f1_mean']:.4f}±{m['cv_f1_std']:.4f}" if m['cv_f1_mean'] else "N/A",
    'time_min': m['train_time_s'] / 60
} for m in all_metrics])

summary_df = summary_df.sort_values('macro_f1', ascending=False)
summary_df.to_csv("models_rigorous/training_summary.csv", index=False)

logger.info(f"\n{summary_df.to_string(index=False)}")

best = summary_df.iloc[0]
logger.info(f"\n🏆 BEST MODEL: {best['model']}")
logger.info(f"   Strategy: {best['strategy']}")
logger.info(f"   Macro F1: {best['macro_f1']:.4f}")
logger.info(f"   Accuracy: {best['accuracy']:.4f}")

logger.info("\n" + "="*80)
logger.info("✅ Pipeline Completed Successfully!")
logger.info("="*80)
logger.info(f"Outputs: models_rigorous/")
logger.info(f"Log: {log_file}")
logger.info("="*80)

NameError: name 'sys' is not defined

In [ ]:
df.head()

,timestamp,ap_id,band,channel,channel_width_mhz,tx_power_dbm,noise_floor_dbm,nwifi_detected,nwifi_type,avg_client_snr_db,throughput_avg_mbps,p95_retry_pct,mean_qoe,mean_distance_m,p95_distance_m,max_distance_m
0,2025-11-02T00:00:00,AP_1,5.0,44,40,17.0,-97.712907,False,wifi,36.898203,321.647555,3.748807,4.892046,6.610379,9.128585,9.852401
1,2025-11-02T00:00:00,AP_2,5.0,36,80,14.0,-96.026344,False,wifi,36.986699,551.080980,4.369926,4.823208,6.120770,9.539560,9.790604
2,2025-11-02T00:00:00,AP_3,5.0,44,20,19.0,-95.451788,False,wifi,35.206611,180.431382,5.932573,4.749566,6.484995,9.660293,9.829710
3,2025-11-02T00:00:00,AP_4,2.4,6,20,16.0,-92.718548,True,FHSS,33.245024,56.037188,21.371680,4.276843,6.650650,9.597728,9.897326
4,2025-11-02T00:00:00,AP_5,5.0,48,40,18.0,-94.064154,False,wifi,34.217565,300.735333,6.852111,4.637710,6.648165,9.677919,9.985162


hierarchical

In [ ]:
"""
Hierarchical Non-Wi-Fi Classifier (AP Telemetry)
================================================
Stage-1: Wi-Fi vs Non-Wi-Fi (binary XGBoost)
Stage-2: BLE / ZigBee / Microwave / FHSS (multi-class XGBoost on non-Wi-Fi only)

Outputs (models_hierarchical/):
  - stage1_wifi_vs_nonwifi.pkl               (model, scaler, label encoder)
  - stage2_nonwifi_multiclass.pkl            (model, scaler, label encoder)
  - loss_*.png, confusion_*.png, importance_*.png
  - training_summary.json (combined metrics)
  - class_distribution.png
  - inference_example.json
"""

# ---------------- CONFIG ----------------

import os, time, json, psutil, logging, warnings
from datetime import datetime
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---------------- LOGGING ----------------
SAVE_DIR = "models_hierarchical"
os.makedirs(SAVE_DIR, exist_ok=True)
log_file = f"{SAVE_DIR}/train_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)  # ✅ ensures it appears in notebook cell output
    ],
    force=True  # ✅ required for Jupyter/Colab
)

logger = logging.getLogger(__name__)

RANDOM_SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

np.random.seed(RANDOM_SEED)

# ---------------- HELPERS ----------------
def save_confusion(cm, classes, title, path):
    fig, ax = plt.subplots(figsize=(9, 7))
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(cmap="Blues", ax=ax, xticks_rotation=45, colorbar=False)
    # add counts + %
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i].sum() == 0:
                pct = 0.0
            else:
                pct = cm[i, j] / cm[i].sum() * 100
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)", ha="center", va="center", color=color, fontsize=9)
    plt.title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

def top_feat_importance(model, feat_cols, title, path, k=15):
    imp = model.feature_importances_
    df = pd.DataFrame({"feature": feat_cols, "importance": imp}).sort_values("importance", ascending=False)
    plt.figure(figsize=(10, 6))
    plt.barh(df["feature"][:k], df["importance"][:k])
    plt.gca().invert_yaxis()
    plt.title(title, fontsize=13, fontweight="bold")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    return df

def training_curve(evals_result, best_iter, title, path):
    """
    Plot XGBoost training curve, handling both binary ('logloss')
    and multi-class ('logloss') metrics gracefully.
    """
    plt.figure(figsize=(9, 5))
    for key, values in evals_result.items():
        metric_name = 'logloss' if 'logloss' in values else 'logloss'
        label = "Train" if key.endswith("0") else "Valid"
        plt.plot(values[metric_name], label=f"{label} ({metric_name})", linewidth=2, alpha=0.85)
    if best_iter is not None:
        plt.axvline(best_iter, color="red", linestyle="--", alpha=0.6, label="Best")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def describe_target(df, col, out_path):
    counts = df[col].value_counts()
    plt.figure(figsize=(10, 5))
    counts.plot(kind="bar", color="steelblue", edgecolor="black")
    plt.title(f"Class Distribution: {col}", fontsize=14, fontweight="bold")
    plt.xlabel("Class"); plt.ylabel("Count")
    plt.xticks(rotation=45)
    for i, v in enumerate(counts.values):
        plt.text(i, v + max(50, 0.01*v), f"{v:,}", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    return counts.to_dict()

def cross_val_macro_f1(X, y, sample_weight, model_params, folds=5):
    """Stratified K-Fold cross-validation that supports both binary and multi-class"""
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import f1_score
    import numpy as np

    n_classes = len(np.unique(y))
    metric = "mlogloss" if n_classes > 2 else "logloss"

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        w_tr = sample_weight[train_idx] if sample_weight is not None else None

        model_params["eval_metric"] = metric  # ✅ set correctly per case
        model = XGBClassifier(**model_params)

        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average="macro"))

    return np.mean(scores), np.std(scores)


# ---------------- LOAD DATA ----------------
logger.info("="*85)
logger.info("HIERARCHICAL TRAINING: Stage-1 (binary) + Stage-2 (multi-class)")
logger.info("="*85)
logger.info(f"Data path: {DATA_PATH}")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

t0 = time.time()
df = pd.read_csv(DATA_PATH)
logger.info(f"Loaded: {len(df):,} rows, {df.shape[1]} cols | Mem={df.memory_usage(deep=True).sum()/1024**2:.2f} MB | {time.time()-t0:.2f}s")

# --- Expect the target column to be 'nwifi_type' with values: wifi / BLE / ZigBee / Microwave / FHSS
if "nwifi_type" not in df.columns:
    raise ValueError("Target column 'nwifi_type' not found in dataset.")

# ---------------- FEATURE LIST ----------------
drop_cols = [c for c in ["timestamp", "ap_id", "nwifi_detected"] if c in df.columns]
target_col = "nwifi_type"

feat_cols = [c for c in df.columns if c not in drop_cols + [target_col]]
logger.info(f"Feature columns ({len(feat_cols)}): {feat_cols}")

# ---------------- STAGE-1 DATA (BINARY) ----------------
df["is_nonwifi"] = (df[target_col] != "wifi").astype(int)
counts1 = describe_target(df, "is_nonwifi", f"{SAVE_DIR}/class_distribution_stage1.png")
logger.info(f"Stage-1 distribution: {counts1}")

X1 = df[feat_cols].fillna(0).copy()
y1 = df["is_nonwifi"].values.astype(int)

# Train/test split for Stage-1
X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=TEST_SIZE, stratify=y1, random_state=RANDOM_SEED
)

# Scale
scaler1 = RobustScaler()
X1_tr_s = scaler1.fit_transform(X1_tr).astype(np.float32)
X1_te_s = scaler1.transform(X1_te).astype(np.float32)

# Class weights for binary
wts1 = compute_class_weight(class_weight="balanced", classes=np.unique(y1_tr), y=y1_tr)
cw1_map = dict(zip(np.unique(y1_tr), wts1))
sw1 = np.asarray([cw1_map[c] for c in y1_tr], dtype=np.float32)

logger.info(f"Stage-1 class weights: {cw1_map}")

# ---------------- STAGE-1 TRAIN ----------------
logger.info("\n[Stage-1] Training Wi-Fi vs Non-Wi-Fi…")
params1 = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_estimators": 2000,
    "learning_rate": 0.05,
    "max_depth": 7,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "gamma": 0.3,
    "reg_lambda": 1.5,
    "min_child_weight": 3,
    "n_jobs": -1
}
t1 = time.time()
model1 = XGBClassifier(**params1)
model1.fit(
    X1_tr_s, y1_tr,
    sample_weight=sw1,
    eval_set=[(X1_tr_s, y1_tr), (X1_te_s, y1_te)],
    callbacks=[EarlyStopping(rounds=50, save_best=True, maximize=False, data_name="validation_1", metric_name="logloss")],
    verbose=False
)
t1_elapsed = time.time() - t1
logger.info(f"[Stage-1] Trained in {t1_elapsed/60:.2f} min. Best iter: {getattr(model1, 'best_iteration', None)}")

# Curves & importance
training_curve(model1.evals_result(), getattr(model1, "best_iteration", None),
               "Stage-1 (Binary) – Training Curve", f"{SAVE_DIR}/loss_stage1.png")
fi1 = top_feat_importance(model1, feat_cols, "Stage-1 – Top Features", f"{SAVE_DIR}/importance_stage1.png")

# Evaluate Stage-1
y1_pred = model1.predict(X1_te_s)
rep1 = classification_report(y1_te, y1_pred, target_names=["wifi", "nonwifi"], output_dict=True, digits=4)
cm1 = confusion_matrix(y1_te, y1_pred)
save_confusion(cm1, ["wifi", "nonwifi"], "Stage-1 – Confusion Matrix", f"{SAVE_DIR}/confusion_stage1.png")
logger.info(f"[Stage-1] Accuracy={rep1['accuracy']:.4f} | Macro-F1={rep1['macro avg']['f1-score']:.4f}")

# Cross-val on Stage-1
cv_mean1, cv_std1 = cross_val_macro_f1(X1_tr_s, y1_tr, sw1, {**params1, "n_estimators": getattr(model1, "best_iteration", 500)}, folds=CV_FOLDS)
logger.info(f"[Stage-1] CV Macro-F1: {cv_mean1:.4f} ± {cv_std1:.4f}")

# Save Stage-1
joblib.dump((model1, scaler1), f"{SAVE_DIR}/stage1_wifi_vs_nonwifi.pkl")

# ---------------- STAGE-2 DATA (MULTI-CLASS, NON-WIFI ONLY) ----------------
df2 = df[df[target_col] != "wifi"].copy()
if df2.empty:
    raise ValueError("No non-Wi-Fi samples found for Stage-2.")

counts2 = describe_target(df2, target_col, f"{SAVE_DIR}/class_distribution_stage2.png")
logger.info(f"Stage-2 distribution: {counts2}")

X2 = df2[feat_cols].fillna(0).copy()
y2 = df2[target_col].astype(str).values

le2 = LabelEncoder()
y2_enc = le2.fit_transform(y2)
classes2 = list(le2.classes_)
logger.info(f"Stage-2 classes: {classes2}")

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2_enc, test_size=TEST_SIZE, stratify=y2_enc, random_state=RANDOM_SEED
)

# Scale
scaler2 = RobustScaler()
X2_tr_s = scaler2.fit_transform(X2_tr).astype(np.float32)
X2_te_s = scaler2.transform(X2_te).astype(np.float32)

# Balance with SMOTE for hard minority classes
logger.info("[Stage-2] Applying SMOTE to balance non-Wi-Fi subclasses…")
smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
X2_tr_bal, y2_tr_bal = smote.fit_resample(X2_tr_s, y2_tr)
X2_tr_bal = np.asarray(X2_tr_bal, dtype=np.float32)
y2_tr_bal = np.asarray(y2_tr_bal, dtype=np.int32)
logger.info(f"[Stage-2] After SMOTE: {len(X2_tr_bal):,} samples")

# Class weights (still useful post-SMOTE for robustness)
wts2 = compute_class_weight(class_weight="balanced", classes=np.unique(y2_tr_bal), y=y2_tr_bal)
cw2_map = dict(zip(np.unique(y2_tr_bal), wts2))
sw2 = np.asarray([cw2_map[c] for c in y2_tr_bal], dtype=np.float32)
logger.info(f"[Stage-2] class weights: { {classes2[k]:float(v) for k,v in cw2_map.items()} }")
# ---------------- STAGE 2: Non-Wi-Fi Subclassifier ----------------
logger.info("\n" + "=" * 80)
logger.info("[Stage-2] Training Non-Wi-Fi Subclassifier (BLE / ZigBee / Microwave / FHSS)")
logger.info("=" * 80)

# Ensure test split is scaled (train set X2_tr_bal is already resampled/ready)
X2_te_s = scaler2.transform(X2_te).astype(np.float32)

params2 = {
    "objective": "multi:softprob",       # multi-class probability output
    "num_class": len(classes2),          # 4 classes
    "eval_metric": "mlogloss",           # specified only here (not again in .fit)
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_estimators": 2500,
    "learning_rate": 0.04,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.5,
    "reg_lambda": 2.0,
    "min_child_weight": 4,
    "n_jobs": -1,
}

# Initialize model
model2 = XGBClassifier(**params2)

# Define evaluation set (train + validation subset for Stage-2)
eval_set_2 = [(X2_tr_bal, y2_tr_bal), (X2_te_s, y2_te)]

# Train with visible progress each 100 rounds
t2 = time.time()
model2.fit(
    X2_tr_bal, y2_tr_bal,
    sample_weight=sw2,
    eval_set=eval_set_2,
    verbose=100
)
t2_elapsed = time.time() - t2
logger.info(f"[Stage-2] Trained in {t2_elapsed/60:.2f} min. Best iter: {getattr(model2, 'best_iteration', None)}")

# Curves & importance
try:
    training_curve(model2.evals_result(), getattr(model2, "best_iteration", None),
                   "Stage-2 (Multi-class) – Training Curve", f"{SAVE_DIR}/loss_stage2.png")
    logger.info(f"[Stage-2] Saved: {SAVE_DIR}/loss_stage2.png")
except Exception as e:
    logger.warning(f"[Stage-2] Could not plot training curves: {e}")

# Use the same feature list as Stage-1 unless you created a Stage-2 variant
feat_cols2 = feat_cols
fi2 = top_feat_importance(model2, feat_cols2, "Stage-2 – Top Features", f"{SAVE_DIR}/importance_stage2.png")
logger.info(f"[Stage-2] Saved: {SAVE_DIR}/importance_stage2.png")

# Evaluate Stage-2 (on *true* non-Wi-Fi test set)
y2_pred = model2.predict(X2_te_s)
rep2 = classification_report(y2_te, y2_pred, target_names=classes2, output_dict=True, digits=4)
cm2 = confusion_matrix(y2_te, y2_pred)
save_confusion(cm2, classes2, "Stage-2 – Confusion Matrix", f"{SAVE_DIR}/confusion_stage2.png")
logger.info(f"[Stage-2] Accuracy={rep2['accuracy']:.4f} | Macro-F1={rep2['macro avg']['f1-score']:.4f}")
logger.info(f"[Stage-2] Per-class:")
for cls in classes2:
    logger.info(f"  {cls:12s} | P={rep2[cls]['precision']:.3f} | R={rep2[cls]['recall']:.3f} | F1={rep2[cls]['f1-score']:.3f}")

# Cross-val on Stage-2
cv_mean2, cv_std2 = cross_val_macro_f1(
    X2_tr_bal, y2_tr_bal, sw2,
    {**params2, "n_estimators": int(getattr(model2, "best_iteration", params2["n_estimators"]))},
    folds=CV_FOLDS
)
logger.info(f"[Stage-2] CV Macro-F1: {cv_mean2:.4f} ± {cv_std2:.4f}")

# Save Stage-2 model bundle
joblib.dump((model2, scaler2, le2), f"{SAVE_DIR}/stage2_nonwifi_multiclass.pkl")
logger.info(f"[Stage-2] Saved model → {SAVE_DIR}/stage2_nonwifi_multiclass.pkl")

# ---------------- END-TO-END (CHAINED) EVALUATION ----------------
logger.info("\n[Chain] Evaluating chained Stage-1 → Stage-2 on a held-out set…")

# Build an overall held-out set from original df
X_all = df[feat_cols].fillna(0).copy().values
y_all = df[target_col].astype(str).values

X_tr_all, X_te_all, y_tr_all, y_te_all = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_SEED
)

# Stage-1 inference
X_te_all_s1 = scaler1.transform(X_te_all).astype(np.float32)
pred_binary = model1.predict(X_te_all_s1)  # 0=wifi, 1=nonwifi

# Stage-2 inference (only for predicted non-Wi-Fi)
final_preds = []
idx_nonwifi = np.where(pred_binary == 1)[0]
sub_labels = np.array([], dtype=str)
if len(idx_nonwifi) > 0:
    X_nonwifi = scaler2.transform(X_te_all[idx_nonwifi]).astype(np.float32)
    sub_preds = model2.predict(X_nonwifi)
    sub_labels = le2.inverse_transform(sub_preds)

j = 0
for i in range(len(X_te_all)):
    if pred_binary[i] == 0:
        final_preds.append("wifi")
    else:
        final_preds.append(sub_labels[j])
        j += 1

final_preds = np.array(final_preds)
all_classes = ["wifi"] + list(classes2)
cm_chain = confusion_matrix(y_te_all, final_preds, labels=all_classes)
save_confusion(cm_chain, all_classes, "Chained Model – Confusion Matrix", f"{SAVE_DIR}/confusion_chained.png")

report_chain = classification_report(y_te_all, final_preds, labels=all_classes, output_dict=True, digits=4)
logger.info(f"[Chain] Accuracy={report_chain['accuracy']:.4f} | Macro-F1={report_chain['macro avg']['f1-score']:.4f}")

# ---------------- STATS & SUMMARY ----------------
proc = psutil.Process(os.getpid())
summary = {
    "data_path": DATA_PATH,
    "n_rows": int(len(df)),
    "n_features": int(len(feat_cols)),
    "stage1": {
        "classes": ["wifi", "nonwifi"],
        "accuracy": float(rep1["accuracy"]),
        "macro_f1": float(rep1["macro avg"]["f1-score"]),
        "cv_macro_f1": float(cv_mean1),
        "cv_std": float(cv_std1),
        "train_time_s": float(t1_elapsed),
        "best_iteration": int(getattr(model1, "best_iteration", -1)),
        "top_features": fi1.head(10).to_dict("records"),
    },
    "stage2": {
        "classes": list(classes2),
        "accuracy": float(rep2["accuracy"]),
        "macro_f1": float(rep2["macro avg"]["f1-score"]),
        "cv_macro_f1": float(cv_mean2),
        "cv_std": float(cv_std2),
        "train_time_s": float(t2_elapsed),
        "best_iteration": int(getattr(model2, "best_iteration", -1)),
        "top_features": fi2.head(10).to_dict("records"),
    },
    "chained_eval": {
        "accuracy": float(report_chain["accuracy"]),
        "macro_f1": float(report_chain["macro avg"]["f1-score"]),
        "per_class": {
            k: {m: float(v) for m, v in d.items()}
            for k, d in report_chain.items()
            if k not in ["accuracy", "macro avg", "weighted avg"]
        },
    },
    "system": {
        "cpu_percent": float(psutil.cpu_percent(interval=None)),
        "ram_mb": float(proc.memory_info().rss / 1024**2),
        "runtime_min": float((time.time() - t0) / 60.0),
        "log_file": log_file
    }
}
os.makedirs(SAVE_DIR, exist_ok=True)
json.dump(summary, open(f"{SAVE_DIR}/training_summary.json", "w"), indent=2)
logger.info("\n=== FINAL SUMMARY ===")
logger.info(json.dumps(summary, indent=2))

# ---------------- SIMPLE INFERENCE WRAPPER ----------------
def chained_infer(ap_row_dict):
    """
    ap_row_dict: single dict of AP telemetry with same feature keys as training.
    Returns: final_label, stage1_prob_nonwifi, (stage2_label, stage2_probs_if_any)
    """
    x = np.array([[ap_row_dict.get(f, 0) for f in feat_cols]], dtype=np.float32)
    x1 = scaler1.transform(x)
    p_nonwifi = float(model1.predict_proba(x1)[0, 1])
    if p_nonwifi < 0.5:
        return "wifi", p_nonwifi, None
    x2 = scaler2.transform(x)
    probs2 = model2.predict_proba(x2)[0]
    idx = int(np.argmax(probs2))
    label2 = le2.inverse_transform([idx])[0]
    return label2, p_nonwifi, {cls: float(probs2[i]) for i, cls in enumerate(classes2)}

# Demo inference (first test row)
example = {k: float(v) for k, v in zip(feat_cols, X_te_all[0])}
pred_label, p_nonwifi, stage2_detail = chained_infer(example)
json.dump(
    {"input_row_sample": example, "pred_label": pred_label, "stage1_p_nonwifi": p_nonwifi, "stage2_detail": stage2_detail},
    open(f"{SAVE_DIR}/inference_example.json", "w"),
    indent=2
)
logger.info("\nSaved chained inference example → models_hierarchical/inference_example.json")
logger.info("\n✅ Done. Artifacts in: models_hierarchical/")



2025-11-03 11:03:47,823 - INFO - =====================================================================================
2025-11-03 11:03:47,824 - INFO - HIERARCHICAL TRAINING: Stage-1 (binary) + Stage-2 (multi-class)
2025-11-03 11:03:47,825 - INFO - =====================================================================================
2025-11-03 11:03:47,826 - INFO - Data path: /content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/ap_log.csv
2025-11-03 11:03:47,998 - INFO - Loaded: 57,540 rows, 16 cols | Mem=14.90 MB | 0.17s
2025-11-03 11:03:48,000 - INFO - Feature columns (12): ['band', 'channel', 'channel_width_mhz', 'tx_power_dbm', 'noise_floor_dbm', 'avg_client_snr_db', 'throughput_avg_mbps', 'p95_retry_pct', 'mean_qoe', 'mean_distance_m', 'p95_distance_m', 'max_distance_m']
2025-11-03 11:03:48,193 - INFO - Stage-1 distribution: {0: 36040, 1: 21500}
2025-11-03 11:03:48,263 - INFO - Stage-1 class weights: {np.int64(0): np.float64(0.7982796892341842), np.int64(1): np.float64(1

<Figure size 900x500 with 0 Axes>

+fft


In [ ]:
"""
FFT-ENHANCED HIERARCHICAL TRAINING
Stage-1: Wi-Fi vs Non-Wi-Fi (binary, AP+FFT features)
Stage-2: BLE / ZigBee / Microwave / FHSS (multiclass, AP+FFT features)

What this script does
---------------------
1) Loads AP & FFT datasets
2) Derives compact FFT features from fft_bin_* columns
3) Merges at (timestamp, ap_id)
4) Builds a clean target 'nwifi_type' with values in:
     ['wifi', 'BLE', 'ZigBee', 'Microwave', 'FHSS']
5) Stage-1 trains Wi-Fi vs Non-Wi-Fi
6) Stage-2 trains 4-class on non-Wi-Fi subset (SMOTE balanced)
7) Saves models, curves, confusion matrices, and a chained evaluation

Outputs directory: models_hierarchical/
"""

import os, time, json, logging, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ======================== CONFIG ========================
AP_CSV_PATH  = "/content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/ap_log.csv"
FFT_CSV_PATH = "/content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/fft_dataset.csv"  # or fft_dataset_24h.csv
SAVE_DIR = "models_hierarchical"
os.makedirs(SAVE_DIR, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5
TARGET_ALLOWED = {"wifi", "BLE", "ZigBee", "Microwave", "FHSS"}

# ======================== LOGGING ========================
log_file = f"{SAVE_DIR}/train_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

# ======================== HELPERS ========================
def plot_training_curve(evals_result: dict, title: str, path: str):
    """Plot training/validation metric curves regardless of metric key naming."""
    try:
        if not evals_result:
            raise ValueError("Empty evals_result")
        # Find first eval set and first metric name
        first_set = next(iter(evals_result.keys()))
        first_metric = next(iter(evals_result[first_set].keys()))
        plt.figure(figsize=(9,5))
        for k, metrics in evals_result.items():
            label = "Train" if k.endswith("0") else "Valid"
            plt.plot(metrics[first_metric], label=f"{label} ({k})", linewidth=2, alpha=0.85)
        plt.title(title)
        plt.xlabel("Iteration")
        plt.ylabel(first_metric)
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()
        logger.info(f"Saved: {path}")
    except Exception as e:
        logger.warning(f"Could not plot training curves: {e}")

def save_confusion(cm: np.ndarray, class_names, title: str, path: str):
    fig, ax = plt.subplots(figsize=(9,7))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(cmap="Blues", ax=ax, colorbar=False, xticks_rotation=45)
    # annotate %
    for i in range(cm.shape[0]):
        row_sum = cm[i].sum() if cm[i].sum() else 1
        for j in range(cm.shape[1]):
            pct = 100.0 * cm[i, j] / row_sum
            color = "white" if cm[i, j] > cm.max()/2 else "black"
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)", ha="center", va="center", color=color, fontsize=9)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    logger.info(f"Saved: {path}")

def cross_val_macro_f1(X, y, sample_weight, model_params, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for fi, (tr, va) in enumerate(skf.split(X, y), 1):
        Xtr, Xva = X[tr], X[va]
        ytr, yva = y[tr], y[va]
        sw = sample_weight[tr] if sample_weight is not None else None
        clf = XGBClassifier(**model_params)
        clf.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva, yva)], verbose=False)
        pred = clf.predict(Xva)
        scores.append(f1_score(yva, pred, average="macro"))
        logger.info(f"  Fold {fi}/{folds} macro-F1: {scores[-1]:.4f}")
    return float(np.mean(scores)), float(np.std(scores))

def normalize_type(val):
    """
    Build a clean single target label:
    - If NaN or 'none' -> 'wifi'
    - If CSV had 1:many FFT rows merged, we still keep the FFT 'label' per row
    - If AP 'nwifi_types' exists and not 'none', take the first token
    """
    if pd.isna(val): return "wifi"
    s = str(val).strip()
    if s.lower() == "none" or s == "nan": return "wifi"
    # If merged brought 'A,B' (shouldn't in latest gen), keep first token
    if "," in s:
        s = s.split(",")[0].strip()
    return s

# ======================== START ========================
logger.info("="*94)
logger.info("FFT-ENHANCED HIERARCHICAL TRAINING: Stage-1 (binary) + Stage-2 (multi-class)")
logger.info("="*94)
logger.info(f"AP file:  {AP_CSV_PATH}")
logger.info(f"FFT file: {FFT_CSV_PATH}")

# ---------- Load ----------
ap = pd.read_csv(AP_CSV_PATH)
fft = pd.read_csv(FFT_CSV_PATH)
logger.info(f"Loaded AP log:  {len(ap):,} rows, {ap.shape[1]} cols")
logger.info(f"Loaded FFT log: {len(fft):,} rows, {fft.shape[1]} cols")

# ---------- Derive FFT features ----------
fft_bin_cols = [c for c in fft.columns if c.startswith("fft_bin_")]
if not fft_bin_cols:
    raise ValueError("No fft_bin_* columns found in FFT dataset.")

bins = fft[fft_bin_cols].values.astype(np.float32)
# basic spectral features
fft_mean = bins.mean(axis=1)
fft_std = bins.std(axis=1)
fft_max = bins.max(axis=1)
fft_min = bins.min(axis=1)
fft_power_sum = (10 ** (bins / 10.0)).sum(axis=1)  # linear power sum
# spectral centroid & bandwidth (index-based)
idx = np.arange(bins.shape[1])[None, :]
weights = (10 ** (bins / 10.0))
wsum = np.clip(weights.sum(axis=1), 1e-9, None)
centroid = (weights * idx).sum(axis=1) / wsum
bandwidth = np.sqrt(((idx - centroid[:, None]) ** 2 * weights).sum(axis=1) / wsum)

fft_feat = pd.DataFrame({
    "timestamp": fft["timestamp"],
    "ap_id": fft["ap_id"],
    "fft_mean": fft_mean,
    "fft_std": fft_std,
    "fft_max": fft_max,
    "fft_min": fft_min,
    "fft_power_sum": fft_power_sum,
    "fft_centroid": centroid,
    "fft_bandwidth": bandwidth,
    # the FFT label will be our primary target when present
    "fft_label": fft.get("label", pd.Series(index=fft.index, dtype=object)),
})
logger.info(f"Generated FFT features: {list(fft_feat.columns.drop(['timestamp','ap_id']))}")

# ---------- Merge AP + FFT feature frame ----------
# Keep only AP columns we need + a target proxy
ap_keep = [
    "timestamp","ap_id","band","channel","channel_width_mhz","tx_power_dbm",
    "noise_floor_dbm","avg_client_snr_db","throughput_avg_mbps",
    "p95_retry_pct","mean_qoe","mean_distance_m","p95_distance_m","max_distance_m",
]
# Some logs might not have all; keep those that exist
ap_keep = [c for c in ap_keep if c in ap.columns]
ap_slim = ap[ap_keep].copy()

# If AP has 'nwifi_types' or 'nwifi_type' keep it (fallback target)
for tcol in ["nwifi_type","nwifi_types"]:
    if tcol in ap.columns:
        ap_slim[tcol] = ap[tcol]
        break

merged = ap_slim.merge(fft_feat, on=["timestamp","ap_id"], how="inner")  # inner ensures aligned samples
logger.info(f"Merged dataset: {len(merged):,} rows, {merged.shape[1]} cols")

# ---------- Build Target 'nwifi_type' robustly ----------
if "fft_label" in merged.columns:
    y_target = merged["fft_label"].apply(normalize_type)
else:
    # fallback to AP side names
    ap_t = merged.get("nwifi_type", merged.get("nwifi_types", pd.Series(index=merged.index)))
    y_target = ap_t.apply(normalize_type)

# Clamp to allowed
y_target = y_target.apply(lambda s: s if s in TARGET_ALLOWED else "wifi")

# ---------- Feature matrix ----------
# Ensure 'band' is numeric (2.4->0, 5->1, 6->2 if present)
def band_to_num(b):
    if pd.isna(b): return 0
    s = str(b)
    if s.startswith("2.4"): return 0
    if s.startswith("5"): return 1
    if s.startswith("6"): return 2
    try:
        return int(float(s))  # already numeric
    except:
        return 0

merged["band"] = merged["band"].apply(band_to_num)

feat_cols = [
    c for c in [
        "band","channel","channel_width_mhz","tx_power_dbm","noise_floor_dbm",
        "avg_client_snr_db","throughput_avg_mbps","p95_retry_pct","mean_qoe",
        "mean_distance_m","p95_distance_m","max_distance_m",
        "fft_mean","fft_std","fft_max","fft_min","fft_power_sum","fft_centroid","fft_bandwidth",
    ] if c in merged.columns
]

logger.info(f"Feature columns ({len(feat_cols)}): {feat_cols}")

X_all = merged[feat_cols].fillna(0).values.astype(np.float32)
y_all = y_target.values.astype(str)

# ======================== STAGE 1: Binary ========================
logger.info("="*80)
logger.info("[Stage-1] Training Wi-Fi vs Non-Wi-Fi")
logger.info("="*80)

y1_bin = np.where(y_all == "wifi", "wifi", "nonwifi")
le1 = LabelEncoder()
y1_enc = le1.fit_transform(y1_bin)  # length == len(X_all), fixes previous mismatch
classes1 = list(le1.classes_)

X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X_all, y1_enc, test_size=TEST_SIZE, stratify=y1_enc, random_state=RANDOM_SEED
)

scaler1 = RobustScaler()
X1_tr_s = scaler1.fit_transform(X1_tr).astype(np.float32)
X1_te_s = scaler1.transform(X1_te).astype(np.float32)

params1 = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.5,
    gamma=0.3,
    n_jobs=-1,
)

t1 = time.time()
model1 = XGBClassifier(**params1)
model1.fit(
    X1_tr_s, y1_tr,
    eval_set=[(X1_tr_s, y1_tr), (X1_te_s, y1_te)],
    verbose=False,
)
t1_elapsed = time.time() - t1
logger.info(f"[Stage-1] Trained in {t1_elapsed/60:.2f} min. Best iter: {getattr(model1,'best_iteration',None)}")

plot_training_curve(model1.evals_result(), "Stage-1 (Binary) – Training Curve", f"{SAVE_DIR}/loss_stage1.png")

y1_pred = model1.predict(X1_te_s)
rep1 = classification_report(y1_te, y1_pred, target_names=classes1, output_dict=True, digits=4)
cm1 = confusion_matrix(y1_te, y1_pred)
save_confusion(cm1, classes1, "Stage-1 – Confusion Matrix", f"{SAVE_DIR}/confusion_stage1.png")
logger.info(f"[Stage-1] Accuracy={rep1['accuracy']:.4f} | Macro-F1={rep1['macro avg']['f1-score']:.4f}")

# Cross-val
cv_mean1, cv_std1 = cross_val_macro_f1(
    X1_tr_s, y1_tr, None,
    {**params1, "n_estimators": getattr(model1, "best_iteration", params1["n_estimators"])},
    folds=CV_FOLDS
)
logger.info(f"[Stage-1] CV Macro-F1: {cv_mean1:.4f} ± {cv_std1:.4f}")

# Save Stage-1
import joblib
joblib.dump((model1, scaler1, le1, feat_cols), f"{SAVE_DIR}/stage1_wifi_vs_nonwifi.pkl")
logger.info(f"[Stage-1] Saved → {SAVE_DIR}/stage1_wifi_vs_nonwifi.pkl")

# ======================== STAGE 2: 4-Class ========================
logger.info("="*80)
logger.info("[Stage-2] Training Non-Wi-Fi Subclassifier (BLE / ZigBee / Microwave / FHSS)")
logger.info("="*80)

mask_nonwifi = (y_all != "wifi")
X2 = X_all[mask_nonwifi]
y2 = y_all[mask_nonwifi]

le2 = LabelEncoder()
y2_enc = le2.fit_transform(y2)
classes2 = list(le2.classes_)
logger.info(f"Stage-2 classes: {classes2}")

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2_enc, test_size=TEST_SIZE, stratify=y2_enc, random_state=RANDOM_SEED
)

scaler2 = RobustScaler()
X2_tr_s = scaler2.fit_transform(X2_tr).astype(np.float32)
X2_te_s = scaler2.transform(X2_te).astype(np.float32)

# SMOTE to balance subclasses
logger.info("[Stage-2] Applying SMOTE to balance subclasses…")
sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
X2_tr_bal, y2_tr_bal = sm.fit_resample(X2_tr_s, y2_tr)
logger.info(f"[Stage-2] After SMOTE: {len(X2_tr_bal):,} samples")

params2 = dict(
    objective="multi:softprob",
    num_class=len(classes2),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_estimators=2500,
    learning_rate=0.04,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.5,
    reg_lambda=2.0,
    min_child_weight=4,
    n_jobs=-1,
)

t2 = time.time()
model2 = XGBClassifier(**params2)
model2.fit(
    X2_tr_bal, y2_tr_bal,
    eval_set=[(X2_tr_bal, y2_tr_bal), (X2_te_s, y2_te)],
    verbose=False,
)
t2_elapsed = time.time() - t2
logger.info(f"[Stage-2] Trained in {t2_elapsed/60:.2f} min. Best iter: {getattr(model2,'best_iteration',None)}")

plot_training_curve(model2.evals_result(), "Stage-2 (Multi-class) – Training Curve", f"{SAVE_DIR}/loss_stage2.png")

y2_pred = model2.predict(X2_te_s)
rep2 = classification_report(y2_te, y2_pred, target_names=classes2, output_dict=True, digits=4)
cm2 = confusion_matrix(y2_te, y2_pred)
save_confusion(cm2, classes2, "Stage-2 – Confusion Matrix", f"{SAVE_DIR}/confusion_stage2.png")
logger.info(f"[Stage-2] Accuracy={rep2['accuracy']:.4f} | Macro-F1={rep2['macro avg']['f1-score']:.4f}")

# CV on Stage-2
cv_mean2, cv_std2 = cross_val_macro_f1(
    X2_tr_bal, y2_tr_bal, None,
    {**params2, "n_estimators": getattr(model2, "best_iteration", params2["n_estimators"])},
    folds=CV_FOLDS
)
logger.info(f"[Stage-2] CV Macro-F1: {cv_mean2:.4f} ± {cv_std2:.4f}")

# Save Stage-2
joblib.dump((model2, scaler2, le2, feat_cols), f"{SAVE_DIR}/stage2_nonwifi_multiclass.pkl")
logger.info(f"[Stage-2] Saved → {SAVE_DIR}/stage2_nonwifi_multiclass.pkl")

# ======================== CHAINED EVAL ========================
logger.info("\n[Chain] Evaluating chained Stage-1 → Stage-2 on a held-out split…")

# fresh split on the full merged data for a fair end-to-end check
X_tr_all, X_te_all, y_tr_all, y_te_all = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_SEED
)

# Run stage-1
X_te_all_s1 = scaler1.transform(X_te_all).astype(np.float32)
pred_bin = model1.predict(X_te_all_s1)  # 0=wifi,1=nonwifi (by label encoder)
bin_labels = le1.inverse_transform(pred_bin)

final_preds = []
idx_nonwifi = np.where(bin_labels == "nonwifi")[0]
sub_labels = np.array([], dtype=str)

if len(idx_nonwifi) > 0:
    X_sub = scaler2.transform(X_te_all[idx_nonwifi]).astype(np.float32)
    sub_pred = model2.predict(X_sub)
    sub_labels = le2.inverse_transform(sub_pred)

j = 0
for i in range(len(X_te_all)):
    if bin_labels[i] == "wifi":
        final_preds.append("wifi")
    else:
        final_preds.append(sub_labels[j] if j < len(sub_labels) else "BLE")  # safe fallback
        j += 1

final_preds = np.array(final_preds)
labels_full = ["wifi"] + classes2
cm_chain = confusion_matrix(y_te_all, final_preds, labels=labels_full)
save_confusion(cm_chain, labels_full, "Chained Model – Confusion Matrix", f"{SAVE_DIR}/confusion_chained.png")

report_chain = classification_report(y_te_all, final_preds, labels=labels_full, output_dict=True, digits=4)
logger.info(f"[Chain] Accuracy={report_chain['accuracy']:.4f} | Macro-F1={report_chain['macro avg']['f1-score']:.4f}")

# ======================== SUMMARY JSON ========================
proc = psutil.Process(os.getpid())
summary = {
    "data": {
        "ap_rows": int(len(ap)),
        "fft_rows": int(len(fft)),
        "merged_rows": int(len(merged)),
        "n_features": int(len(feat_cols))
    },
    "stage1": {
        "classes": classes1,
        "accuracy": float(rep1["accuracy"]),
        "macro_f1": float(rep1["macro avg"]["f1-score"]),
        "cv_macro_f1": float(cv_mean1),
        "cv_std": float(cv_std1),
        "train_time_s": float(t1_elapsed),
        "best_iteration": int(getattr(model1, "best_iteration", -1))
    },
    "stage2": {
        "classes": classes2,
        "accuracy": float(rep2["accuracy"]),
        "macro_f1": float(rep2["macro avg"]["f1-score"]),
        "cv_macro_f1": float(cv_mean2),
        "cv_std": float(cv_std2),
        "train_time_s": float(t2_elapsed),
        "best_iteration": int(getattr(model2, "best_iteration", -1))
    },
    "chained": {
        "accuracy": float(report_chain["accuracy"]),
        "macro_f1": float(report_chain["macro avg"]["f1-score"]),
    },
    "system": {
        "cpu_percent": float(psutil.cpu_percent(interval=None)),
        "ram_mb": float(proc.memory_info().rss / 1024**2),
        "runtime_min": float((time.time() - psutil.Process(os.getpid()).create_time())/60.0),
        "log_file": log_file
    }
}
with open(f"{SAVE_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
logger.info("\n=== FINAL SUMMARY ===")
logger.info(json.dumps(summary, indent=2))

# ======================== INFERENCE WRAPPER ========================
def chained_infer(ap_row: dict):
    """
    ap_row: dict with the same feature keys in feat_cols
    returns: final_label, p_nonwifi, stage2_detail (None if predicted wifi)
    """
    x = np.array([[ap_row.get(k, 0) for k in feat_cols]], dtype=np.float32)
    x1 = scaler1.transform(x)
    p_nonwifi = float(model1.predict_proba(x1)[0, 1])
    if p_nonwifi < 0.5:
        return "wifi", p_nonwifi, None
    x2 = scaler2.transform(x)
    p2 = model2.predict_proba(x2)[0]
    idx = int(np.argmax(p2))
    label2 = le2.inverse_transform([idx])[0]
    return label2, p_nonwifi, {cls: float(p2[i]) for i, cls in enumerate(classes2)}

# quick demo
example_row = dict(zip(feat_cols, X_all[0]))
pred_label, p_nonwifi, stage2_detail = chained_infer(example_row)
with open(f"{SAVE_DIR}/inference_example.json", "w") as f:
    json.dump({
        "input_sample": {k: float(v) for k, v in example_row.items()},
        "pred_label": pred_label,
        "p_nonwifi": p_nonwifi,
        "stage2_detail": stage2_detail
    }, f, indent=2)

logger.info(f"\nSaved chained inference example → {SAVE_DIR}/inference_example.json")
logger.info(f"\n✅ Done. Artifacts in: {SAVE_DIR}/")


2025-11-03 11:22:09,268 - INFO - ==============================================================================================
2025-11-03 11:22:09,269 - INFO - FFT-ENHANCED HIERARCHICAL TRAINING: Stage-1 (binary) + Stage-2 (multi-class)
2025-11-03 11:22:09,281 - INFO - ==============================================================================================
2025-11-03 11:22:09,288 - INFO - AP file:  /content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/ap_log.csv
2025-11-03 11:22:09,292 - INFO - FFT file: /content/drive/MyDrive/RRM+ Arista/rrmplus_24h_20251102_174310/fft_dataset.csv
2025-11-03 11:22:12,117 - INFO - Loaded AP log:  57,540 rows, 16 cols
2025-11-03 11:22:12,118 - INFO - Loaded FFT log: 57,540 rows, 73 cols
2025-11-03 11:22:12,235 - INFO - Generated FFT features: ['fft_mean', 'fft_std', 'fft_max', 'fft_min', 'fft_power_sum', 'fft_centroid', 'fft_bandwidth', 'fft_label']
2025-11-03 11:22:12,392 - INFO - Merged dataset: 70,842 rows, 23 cols
2025-11-03 11:22:12

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score,
    cohen_kappa_score, matthews_corrcoef, f1_score
)
from sklearn.preprocessing import label_binarize
import numpy as np, time, psutil, json

# ---- Extra Metrics (Stage-2) ----
y_true = y2_te
y_pred = model2.predict(X2_te_s)
y_prob = model2.predict_proba(X2_te_s)
classes = range(len(classes2))

extra2 = {
    "precision_macro": precision_score(y_true, y_pred, average="macro"),
    "recall_macro": recall_score(y_true, y_pred, average="macro"),
    "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    "roc_auc_macro": roc_auc_score(
        label_binarize(y_true, classes=classes), y_prob, average="macro"
    ),
    "kappa": cohen_kappa_score(y_true, y_pred),
    "mcc": matthews_corrcoef(y_true, y_pred),
}

# Inference latency
t_infer = []
for i in range(500):
    x = X2_te_s[i % len(X2_te_s)].reshape(1, -1)
    t0 = time.perf_counter(); model2.predict(x); t_infer.append(time.perf_counter() - t0)
extra2["mean_latency_ms"] = float(np.mean(t_infer) * 1000)

proc = psutil.Process(os.getpid())
extra2["ram_mb"] = float(proc.memory_info().rss / 1024**2)
extra2["cpu_percent"] = float(psutil.cpu_percent(interval=None))

# Attach to summary JSON
summary["stage2"]["extra_metrics"] = extra2
json.dump(summary, open(f"{SAVE_DIR}/training_summary_full.json", "w"), indent=2)
logger.info(f"[Stage-2] Extra metrics:\n{json.dumps(extra2, indent=2)}")


2025-11-03 11:32:52,703 - INFO - [Stage-2] Extra metrics:
{
  "precision_macro": 0.9885732922929853,
  "recall_macro": 0.9828706545274751,
  "weighted_f1": 0.9862625557908175,
  "roc_auc_macro": 0.9983186682625381,
  "kappa": 0.9816040397199965,
  "mcc": 0.9818173465287164,
  "mean_latency_ms": 2.724590286010425,
  "ram_mb": 817.6640625,
  "cpu_percent": 13.2
}


In [ ]:
import os
import shutil
from google.colab import drive

# === Step 1: Mount Drive ===
drive.mount('/content/drive')

# === Step 2: Define target folder in Drive ===
TARGET_DIR = "/content/drive/MyDrive/RRM+ Arista/wifi_nonwifi"  # <-- change this as needed
os.makedirs(TARGET_DIR, exist_ok=True)

# === Step 3: Iterate through /content and move folders ===
SRC_DIR = "/content"

for item in os.listdir(SRC_DIR):
    item_path = os.path.join(SRC_DIR, item)

    # Skip the 'drive' folder and non-folders
    if item == "drive" or not os.path.isdir(item_path):
        continue

    dest_path = os.path.join(TARGET_DIR, item)

    # If a folder with the same name exists, append a suffix
    if os.path.exists(dest_path):
        base_name = os.path.basename(item)
        dest_path = os.path.join(TARGET_DIR, f"{base_name}_copy")

    print(f"📦 Moving: {item_path} → {dest_path}")
    shutil.move(item_path, dest_path)

print("\n✅ All non-Drive folders have been moved to:", TARGET_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Moving: /content/.config → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/.config
📦 Moving: /content/models_rigorous → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/models_rigorous
📦 Moving: /content/models_hierarchical → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/models_hierarchical
📦 Moving: /content/models_hierarchical_fft → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/models_hierarchical_fft
📦 Moving: /content/models → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/models
📦 Moving: /content/sample_data → /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi/sample_data

✅ All non-Drive folders have been moved to: /content/drive/MyDrive/RRM+ Arista/wifi_nonwifi
